# Prompts Embedding

## Imports

In [3]:
!pip install -U sentence-transformers datasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import transformers
import random
from sentence_transformers import SentenceTransformer

import warnings
warnings.filterwarnings('ignore')


images_dir = "/content/drive/MyDrive/Data/images_128x128_new.zip"
captions_path = "/content/drive/MyDrive/Data/logo_image_caption.tsv"
output_dir = "/content/drive/MyDrive/logo_recognition_similarity_search_project/output"
os.makedirs(output_dir, exist_ok=True)


## Data Exploration


### Laod dataset


In [5]:
raw_df = pd.read_csv(captions_path, sep='\t')
raw_df.id = raw_df.id.apply(lambda x: str(x).strip().lower())
raw_df.head()

,id,path,prompt
0,1068157504917352548,images/8b/5d/1068157504917352548_0_0.png,an Executive Agenda logo containing a pen
1,1068157504917352548,images/8b/5d/1068157504917352548_0_1.png,an Executive Agenda logo containing a pen
2,1068157504917352548,images/8b/5d/1068157504917352548_1_0.png,an Executive Agenda logo containing a pen
3,1068157504917352548,images/8b/5d/1068157504917352548_1_1.png,an Executive Agenda logo containing a pen
4,1068157546126377001,images/18/ef/1068157546126377001_0_0.png,minimalistic flat logo for private equity firm...


### Exploratory & Preprocessing

In [ ]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1777584 entries, 0 to 1777583
Data columns (total 3 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   id      object
 1   path    object
 2   prompt  object
dtypes: object(3)
memory usage: 40.7+ MB


In [ ]:
raw_df.describe(include='all').T

,count,unique,top,freq
id,1777584,444396,1068157504917352548,4
path,1777584,1777584,images/8b/5d/1068157504917352548_0_0.png,1
prompt,1777584,282276,logo,732


In [ ]:
import re

def clean_text(text):
  """
    Cleans a given text by removing non-english characters. Converts text to lowercase.

    Args:
        text (str): The input text.

    Returns:
        str: The cleaned text. Returns an empty string if input is not a string.
    """

  if not isinstance(text, str):
    return ""
  text = re.sub(r'[^\x00-\x7F]+', ' ', text)
  text = text.lower()
  text = text.strip()

  return text

In [ ]:
raw_df['prompt'] = raw_df['prompt'].progress_apply(clean_text)

100%|██████████| 1777584/1777584 [00:08<00:00, 218027.51it/s]


In [ ]:
raw_df.describe(include='all').T

,count,unique,top,freq
id,1777584,444396,1068157504917352548,4
path,1777584,1777584,images/8b/5d/1068157504917352548_0_0.png,1
prompt,1777584,281151,logo,1256


In [ ]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1777584 entries, 0 to 1777583
Data columns (total 3 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   id      object
 1   path    object
 2   prompt  object
dtypes: object(3)
memory usage: 40.7+ MB


In [ ]:
raw_df.head()

,id,path,prompt
0,1068157504917352548,images/8b/5d/1068157504917352548_0_0.png,an executive agenda logo containing a pen
1,1068157504917352548,images/8b/5d/1068157504917352548_0_1.png,an executive agenda logo containing a pen
2,1068157504917352548,images/8b/5d/1068157504917352548_1_0.png,an executive agenda logo containing a pen
3,1068157504917352548,images/8b/5d/1068157504917352548_1_1.png,an executive agenda logo containing a pen
4,1068157546126377001,images/18/ef/1068157546126377001_0_0.png,minimalistic flat logo for private equity firm...


## Prompts Embedding

In [ ]:
from sentence_transformers import SentenceTransformer
import datasets
import torch

model_id = 'sentence-transformers/all-MiniLM-L6-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# convert raw_df to datasets
ds = datasets.Dataset.from_pandas(raw_df)
ds

Dataset({
    features: ['id', 'path', 'prompt'],
    num_rows: 1777584
})

In [ ]:
# Load model from HuggingFace Hub
model = SentenceTransformer(model_id,device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [ ]:
def batch_encode(examples):
  encoded_prompts = model.encode(examples["prompt"], show_progress_bar=False)
  return {'encoded_prompt': encoded_prompts}

In [ ]:
encoded_prompts_ds = ds.map(
                            batch_encode,
                            batched=True,
                            batch_size=1000,
                            desc="Encoding prompts"
                            )

Encoding prompts:   0%|          | 0/1777584 [00:00<?, ? examples/s]

In [ ]:
# Save the encoded dataset
encoded_prompts_ds.save_to_disk(os.path.join(output_dir, "encoded_prompts_dataset"))

Saving the dataset (0/7 shards):   0%|          | 0/1777584 [00:00<?, ? examples/s]

In [ ]:
encoded_prompts_ds

Dataset({
    features: ['id', 'path', 'prompt', 'encoded_prompt'],
    num_rows: 1777584
})

## Load Encoded Prompts Dataset

In [ ]:
import datasets
import os

ds_path = os.path.join(output_dir, "encoded_prompts_dataset")

ds = datasets.load_from_disk(ds_path)
ds

Dataset({
    features: ['id', 'path', 'prompt', 'encoded_prompt'],
    num_rows: 1777584
})

In [ ]:
print("Shape of the dataset: ", ds.shape)

Shape of the dataset:  (1777584, 4)


In [ ]:
print(ds[0])

{'id': '1068157504917352548', 'path': 'images/8b/5d/1068157504917352548_0_0.png', 'prompt': 'an executive agenda logo containing a pen', 'encoded_prompt': [0.02813463844358921, 0.08543377369642258, 0.06893604248762131, -0.003884686157107353, 0.0654783844947815, 0.005106073804199696, 0.061847977340221405, -0.033956725150346756, 0.12158887833356857, 0.007636446971446276, -0.015827639028429985, 0.07711935043334961, -0.09344757348299026, 0.019994154572486877, 0.0006775992806069553, -0.04366626590490341, -0.04798804968595505, -0.10249566286802292, -0.0020336564630270004, 0.031133590266108513, 0.052370306104421616, 0.006414209492504597, 0.052758194506168365, -0.03269697725772858, -0.05180355906486511, 0.07506828755140305, -0.013911547139286995, -0.02790258079767227, -0.0020461943931877613, -0.03008561208844185, 0.07808278501033783, -0.04226216673851013, 0.02046051062643528, -0.014623149298131466, 0.08209598809480667, 0.002786512253805995, 0.03839430958032608, 0.013472510501742363, 0.08748301

In [ ]:
len(ds[0]['encoded_prompt'])

384

In [ ]:
# convert dataset to DataFrame
df = ds.to_pandas()
df.head()

,id,path,prompt,encoded_prompt
0,1068157504917352548,images/8b/5d/1068157504917352548_0_0.png,an executive agenda logo containing a pen,"[0.028134638, 0.08543377, 0.06893604, -0.00388..."
1,1068157504917352548,images/8b/5d/1068157504917352548_0_1.png,an executive agenda logo containing a pen,"[0.028134638, 0.08543377, 0.06893604, -0.00388..."
2,1068157504917352548,images/8b/5d/1068157504917352548_1_0.png,an executive agenda logo containing a pen,"[0.028134638, 0.08543377, 0.06893604, -0.00388..."
3,1068157504917352548,images/8b/5d/1068157504917352548_1_1.png,an executive agenda logo containing a pen,"[0.028134638, 0.08543377, 0.06893604, -0.00388..."
4,1068157546126377001,images/18/ef/1068157546126377001_0_0.png,minimalistic flat logo for private equity firm...,"[0.084999934, 0.04874548, -0.020015288, -0.048..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1777584 entries, 0 to 1777583
Data columns (total 4 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   id              object
 1   path            object
 2   prompt          object
 3   encoded_prompt  object
dtypes: object(4)
memory usage: 54.2+ MB


In [ ]:
print("Shape of the dataset: ", df.shape)

Shape of the dataset:  (1777584, 4)




---

